# Experiment 06: k Scaling (Normalized Difficulty)

Increase k while also scaling lag and bottleneck with k.

In [1]:
from pathlib import Path
import sys


def find_repo_root(start: Path | None = None) -> Path:
    current = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src" / "config.py").exists() and (candidate / "README.md").exists():
            return candidate
    raise RuntimeError("Could not locate repository root.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")


Repo root: /Users/eomjeonghyeon/Documents/github_project/python-neural-network-research


In [2]:
import numpy as np
import pandas as pd

from src import ExperimentConfig, run_experiment


def overall_row(results: dict) -> pd.Series:
    summary_df = results["summary_df"]
    return summary_df.loc[summary_df["set_idx"] == "overall"].iloc[0]


def collect_overall_rows(result_map: dict[str, dict], columns: list[str] | None = None) -> pd.DataFrame:
    rows = []
    for label, results in result_map.items():
        row = overall_row(results).copy()
        row["run_label"] = label
        rows.append(row)
    df = pd.DataFrame(rows)
    if columns is None:
        return df
    keep = [col for col in ["run_label", *columns] if col in df.columns]
    return df[keep]


In [3]:
def safe_min_delta_theta(base_config: dict, num_freqs: int, slack: float = 0.98, requested: float = 0.04 * np.pi) -> float:
    requested = base_config.get("MIN_DELTA_THETA", requested)
    if num_freqs <= 1:
        return requested
    available_span = base_config["theta_max"] - base_config["theta_min"]
    max_feasible = slack * available_span / (num_freqs - 1)
    return min(requested, max_feasible)


BASE_CONFIG = dict(
    time_mode="discrete",
    theta_min=0.05 * np.pi,
    theta_max=0.85 * np.pi,
    RANDOM_AMPLITUDE=False,
    RANDOM_PHASE=True,
    USE_NOISE=False,
    HIDDEN_DIM=128,
    NUM_EXPERIMENTS=20,
    SEEDS_PER_FREQ=5,
    VAL_RATIO=0.2,
    TEST_RATIO=0.2,
    NORMALIZE_H_COLUMNS=False,
    VERBOSE=False,
    MAKE_PLOTS=False,
)


In [ ]:
results = {}
for num_freqs in [1, 2, 4, 8, 16]:
    seq_len = max(4096, 512 * num_freqs)
    for bottleneck_multiplier in [2, 4]:
        for model_id, lr in [("AN003_LINEAR", 1e-2), ("AN002_NO_BN_TANH", 1e-3)]:
            cfg = ExperimentConfig(
                **BASE_CONFIG,
                SEQ_LEN=seq_len,
                NUM_FREQS=num_freqs,
                MIN_DELTA_THETA=safe_min_delta_theta(BASE_CONFIG, num_freqs),
                LAG=4 * num_freqs,
                BOTTLENECK_MULTIPLIER=bottleneck_multiplier,
                MODEL_ID=model_id,
                LR=lr,
                EPOCHS=1500,
            )
            label = f"model={model_id},k={num_freqs},L={4 * num_freqs},bmul={bottleneck_multiplier}"
            results[label] = run_experiment(cfg)

collect_overall_rows(
    results,
    columns=[
        "test_mse_mean",
        "test_align_coverage_full_mean",
        "test_recon_r2_qf_from_h_mean",
    ],
)


/Users/eomjeonghyeon/Documents/github_project/python-neural-network-research/src/experiment_runner.py:433: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([summary_df, pd.DataFrame([overall_row])], ignore_index=True)
/Users/eomjeonghyeon/Documents/github_project/python-neural-network-research/src/experiment_runner.py:433: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([summary_df, pd.DataFrame([overall_row])], ignore_index=True)
/Users/eomjeonghyeon/Documents/githu